# Joining complaint data: 2011-2020, 2019, and 2020-2026

- Combines the three complaint files into one dataset
- Cleans, standardizes allegation labels, and collapses to one row per case
- Saves the full dataset and a 15-case sample (CSV and JSON) for the LLM pass

## 1. Import and load

In [ ]:
import pandas as pd

df_2011_2020 = pd.read_csv("Data/iad_extracted_with_youth_labels_scrubbed.csv")
df_2019 = pd.read_csv("Data/cleaned_complaints_2019.csv")
df_2020_2026 = pd.read_csv("Data/2020_2026_exploded.csv")

## 2. Handle 2020 cases

The 2011-2020 file contains 2020 cases too. We take those out, then add back only the 2020 cases that are not already in the combined data.

In [ ]:
is_2020 = df_2011_2020["ia_no"].str.startswith("IAD2020", na=False)
df_2020_from_old = df_2011_2020[is_2020]
df_2011_2020 = df_2011_2020[~is_2020]

df_complaints = pd.concat([df_2011_2020, df_2019, df_2020_2026], ignore_index=True)

extra_2020 = df_2020_from_old[~df_2020_from_old["ia_no"].isin(df_complaints["ia_no"])]
df_complaints = pd.concat([df_complaints, extra_2020], ignore_index=True)

## 3. Drop incomplete rows and duplicates

In [ ]:
df_complaints = df_complaints.dropna(subset=["ia_no", "allegation_x", "narrative_x"]).copy()
df_complaints = df_complaints.drop_duplicates().sort_values("ia_no").reset_index(drop=True)

df_complaints.to_csv("Data/complaints_combined_deduped.csv", index=False)
print("Rows:", len(df_complaints))
print("Rows from 2020 cases:", df_complaints["ia_no"].str.startswith("IAD2020").sum())

## 4. Standardize allegation labels

Each raw allegation spelling is mapped to one standard label. Raw values not in the mapping are labeled `UNMAPPED` and printed below, so the dictionary can be extended.

The bias flag is computed before mapping, so it keeps the "bias" wording from the raw label.

In [ ]:
# standardized label -> every raw variant (lowercase) that should collapse into it
GROUPS = {
    "Respectful Treatment": ["respectful treatment", "respectful treatment (2 counts)", "respectful treatment (3 counts)", "respectful treatment bias", "nrespectful treatment"],
    "Neglect of Duty / Unreasonable Judgment": ["neg.duty/unreasonable judge", "neg.duty/unreasonable judge bias", "neg.duty/unreasonable judge (2 counts)", "neg.duty/unreasonable judge (2 counts", "negduty/unreasonable judge", "inegduty/unreasonable judge", "unreasonable judgment bias", "neglect of duty", "negelct of duty", "neglect of duty/unreasonable judgement", "neglect of duty/unreasonable judgment (2 counts)", "neglect of duty/unreasonable judgment (3 counts)", "neglect of duty/unreasonable judgment (4 counts)", "neglect of duty/unreasonable judgment (11 counts)"],
    "Use of Force": ["use of force", "use of force (2 counts)"],
    "Investigation of Use of Force": ["investigation of use of force"],
    "Conduct Unbecoming": ["conduct unbecoming", "conduct unbecoming (2 counts)", "conduct unbecoming (3 counts)", "conduct unbecoming (7 counts)", "rule 102 sec. 3 - conduct"],
    "Conformance to Laws": ["conformance to laws", "conformance to laws (2 counts)", "conformance to laws (2counts)", "conformance to laws (3 counts)", "conformance to laws (11 counts)", "conformance to laws (7 counts)", "rule 102 sec. 35 - conformance to laws"],
    "Self Identification": ["self identification"],
    "Display of Identification": ["display of identification"],
    "BIAS-Free Policing Policy": ["bias-free policing policy"],
    "Abuse of Process": ["abuse of process"],
    "Abuse of Discretion": ["abuse of discretion"],
    "Police Service": ["police service"],
    "Manner of Recording Complaints": ["manner of recording complaints"],
    "Strip Search / Visual Body Cavity Search": ["strip search and or visual body cavity search"],
    "Search Suspect After Handcuffing": ["search suspect after handcuffing"],
    "Prisoner Handling / Booking Procedures": ["prisoners/booking procedures", "booking procedures", "handcuffs", "prisoners", "prisoners - cell inspection", "prisoner inspection", "prisoner - sickness or injury"],
    "Untruthfulness / Inaccurate Reporting": ["untruthfulness", "untruthfulness (2 counts)", "untruthfulness (3 counts)", "untruthfulness in department report", "untruthfulness in department reports (2 counts)", "untruthfulness in department reports (3 counts)", "untruthfulness in department reports (4 counts)", "knowingly entering inaccurate false or improper information", "department reports-truthfulness", "department reports-inacurrate reporting"],
    "Accountability": ["accountability"],
    "Improper Statements": ["improper statements"],
    "Statement of Opinion": ["statement of opinion (6 counts)"],
    "Request for Interviews": ["request for interviews (6 counts)"],
    "Canon Seven": ["canon seven"],
    "Canon Eight": ["canon eight"],
    "CJIS / CORI Access": ["cjis access", "cjis access (4 counts)", "cori access", "cori violation", "cori procedures"],
    "Association with Criminals": ["association with criminals", "association with criminals (2 counts)"],
    "Personal Business": ["personal business"],
    "Evidence & Found Property": ["evidence & found property"],
    "Safeguarding Non-Evidence Property": ["safeguarding non-evidence property"],
    "Handling & Disposition of Seized Money": ["handling & disposistion of seized money"],
    "Release of Property in Police Custody": ["release of property in police custody"],
    "Courts": ["courts"],
    "Courts - Failure to Appear When Summonsed": ["courts - failure to appear when summonsed (4 counts)"],
    "Alcohol on Duty": ["alcohol on duty", "alcohol on duty (2 counts)", "alcohol on duty (3 counts)"],
    "Alcohol in Police Facility": ["alcohol in police facility", "alcohol in police facility (2 counts)", "alcohol in police facility (3 counts)"],
    "Alcohol off Duty": ["alcohol off duty", "alcohol off duty (5 counts)", "rule 102 sec. 14 - use of alcohol off duty"],
    "Drug Testing / Controlled Substances": ["drug test", "drug test (2 counts)", "substance abuse policy", "controlled substances", "controlled substance notification (8 counts)", "under the influence", "prohibited conduct"],
    "Tobacco on Duty": ["tabacco on duty"],
    "Reporting for Duty": ["reporting for duty", "reporting for duty (2 counts)", "reporting for duty (3 counts)", "officer unable to report for duty"],
    "Paid Details": ["paid details", "paid details fitness for duty", "detail cards", "authorized details", "paid detail authorized vendor", "acceptance of details", "acceptance of details (2 counts)", "paid detail assignment"]
             + [f"paid detail assignment ({n} counts)" for n in (3, 4, 8, 16, 17, 18, 20, 21, 23, 41, 44, 71, 89, 98, 131)],
    "Directives / Orders": ["directives/orders", "directives/orders (3 counts)", "directives/orders (5 counts)", "directives & orders (2 counts)", "directives & orders (3 counts)"],
    "Failure to Notify Supervisor When Leaving Assigned Post": ["failure to notify supervisor when leaving their assigned post"],
    "P.O. Notifying Operations": ["p.o. notifying operations (3 counts)", "p.o. notifying operations (9 counts)"],
    "Sleeping on Duty": ["sleeping on duty"],
    "Sick Leave": ["sick leave"],
    "Leaves of Absence": ["leaves of absence"],
    "Off Duty Employment w/out Permission": ["off duty employment w/out permission"],
    "Residency": ["residency"],
    "Change of Address Notification Violation": ["change of address notification violation"],
    "Residence and Telephone": ["residence and telephone"],
    "Uniform & Equipment": ["uniforms and equipment", "uniform specifications", "uniform & equipment-care & maintenance", "uniform & equipment - shoes", "uniform & equipment - trousers"],
    "Body-Worn Camera Policy": ["special order 16-023 section 2.5 bwc notice of recording", "body worn camera policy - categorization", "body worn policy - activation & incidents of use", "bwc activation & incidents of use"],
    "Vehicle Operation / Accidents": ["emergency driving", "emergency driving weather & enviornmental conditions", "dept. motor vehicle accidents", "department motor vehicle accidents notifications"],
    "Permissible Weapons & Ammunition": ["permissible weapons & ammunition"],
    "Reporting Firearms Discharge": ["reporting firearms dishcarge"],
    "Securing and Maintenance of Firearm": ["securing and maintenance of firearm"],
    "Failing to Submit Reports in Timely Manner": ["failing to submit reports in timely manner"],
    "Duty Supervisor": ["duty supervisor"],
    "Area Commanders": ["area commanders"],
    "Special Order 99-48 Dept E-Mail and Computer Use": ["special order 99-48 dept e-mail and computer use"],
    "Violations of Rules & Procedures": ["violations of rules & procedures"],
    "Failure to Be Knowledgeable & Informed": ["failure to be knowledgeable & informed"],
    "Mutual Protection": ["mutual protection"],
    "Information Dept. Members May Not Release": ["information dep.o. members may not release"],
    "Criminal Complaints": ["criminal complaints"],
    "Radio Procedures": ["radio procedures"],
    "F.I.O Reports": ["f.i.o reports"],
    "Completion of FIOE Report": ["completion of fioe report"],
    "Harassment Policy / Sexual Harassment": ["harassment policy", "harassment policy - confidentiality", "harassment policy - retaliation", "harassment policy - retaliation (2 counts)", "harassment policy - retaliation (3 counts)", "sexual harassment - supervisor responsibilities", "sexual harassment - unlawful conduct/harassment"],
    "Hostile Work Environment": ["hostile work environment", "hostile work environment (6 counts)"],
    "Situations Involving Family / Domestic": ["situations involving family & frien", "situations involving family members", "domestic situations involving dep.o. employees"],
    "Situations Involving Off-Duty Boston P.O.s": ["situations involving off-duty boston p.o.s"],
}

# flip into a plain dictionary: raw lowercase label -> standard label
allegation_map = {raw: clean for clean, raws in GROUPS.items() for raw in raws}

# flag bias wording BEFORE mapping
df_complaints["bias_label"] = df_complaints["allegation_x"].str.contains("bias", case=False, na=False)

key = (df_complaints["allegation_x"].astype(str).str.strip().str.lower()
       .str.replace(r"\s+", " ", regex=True))
df_complaints["allegation_standardized"] = key.map(allegation_map).fillna("UNMAPPED")

print("Unmapped raw labels (add these to GROUPS if they are real categories):")
print(df_complaints.loc[df_complaints["allegation_standardized"] == "UNMAPPED", "allegation_x"]
      .value_counts().head(20))

## 5. Collapse to one row per case

Each case (`ia_no`) can have several allegations. We join the unique values with ` | ` so each case is one row. The bias flag becomes True if any allegation in the case has bias wording.

In [ ]:
def join_unique(values):
    cleaned = [str(v).strip() for v in values.dropna()]
    unique = [v for v in dict.fromkeys(cleaned) if v]  # keep order, drop repeats and blanks
    return " | ".join(unique) if unique else None

agg_rules = {col: join_unique for col in df_complaints.columns if col not in ["ia_no", "bias_label"]}
agg_rules["bias_label"] = "any"

df_collapsed = df_complaints.groupby("ia_no", as_index=False).agg(agg_rules)
df_collapsed.to_csv("Data/all_complaints_collapsed.csv", index=False)
print("Cases:", len(df_collapsed))

## 6. Make the 15-case sample

In [ ]:
df_sample = df_collapsed.sample(n=15, random_state=42)
df_sample.to_csv("Data/sample_complaints.csv", index=False)

## 7. Prepare the LLM files

We remove the outcome and category columns so the LLM reads only the complainant's narrative. This keeps the coding blind to findings and dispositions.

In [ ]:
columns_to_drop = [
    "incident_type_x", "received_date_x", "occurred_date_x", "allegation_x",
    "finding_x", "disposition_x", "label", "narrative_y",
    "Off: Title/rank", "Off: First name", "Off: Last name",
    "Act: Action taken", "Action taken date", "Days or hours suspended",
    "Disposition", "Priority", "Occurred date", "clean_name",
    "Off: Employee ID Number", "Act: Days/hours suspended",
    "allegation_standardized", "bias_label",
]

df_llm = df_collapsed.drop(columns=columns_to_drop, errors="ignore")
df_sample_llm = df_sample.drop(columns=columns_to_drop, errors="ignore")

df_llm.to_json("Data/llm_final_df.json", orient="records", indent=2)
df_sample_llm.to_json("Data/sample_complaints_llm.json", orient="records", indent=2)
df_sample_llm.to_csv("Data/sample_complaints_llm.csv", index=False)
df_llm.to_csv("Data/llm_final_df.csv", index=False)

print("LLM columns:", list(df_llm.columns))

## 8. Total word count (LLM dataset)

In [ ]:
def count_total_words(df: pd.DataFrame) -> int:
    total = 0
    for value in df.astype(str).values.ravel():
        total += len(value.split())
    return total

print(f"Total words: {count_total_words(df_llm)}")